# 17 — Phase 5: Backward in C (MLA first)

**Before:** notebooks 15–16 (train + sample in PyTorch; C forward in phase 4).

**Goal:** Understand how `train_gpt2.c` runs **backward** — we add the same for DeepSeek-V2 **piece by piece**.

## What exists now

| Piece | Forward | Backward |
|-------|---------|----------|
| RMSNorm | `rmsnorm.c` | `ds4_rmsnorm_backward` |
| MLA | `mla.c` | `dsv2_mla_backward` |
| MoE | `moe.c` | Phase 5b (TODO) |
| 1-layer train | `train_v2_tiny -train-1layer` | MLA + wte (MoE frozen) |

## Try it

```bash
cd c
./bin/train_v2_tiny -train-1layer 50
```

Compare to llm.c: search `attention_backward` in `vendor/llm.c/train_gpt2.c`.


In [ ]:
# PyTorch reference: MLA backward via autograd (same math C implements)
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention

cfg = DeepSeekV2Config.tiny(64, 16)
attn = MultiHeadLatentAttention(cfg)
x = torch.randn(1, 8, cfg.n_embd, requires_grad=True)
y = attn(x)
y.sum().backward()
print("x.grad norm:", x.grad.norm().item())
print("wq.grad norm:", attn.wq.weight.grad.norm().item())


## Read order (C)

1. `deepseek_v2/ops.c` — linear + softmax backward
2. `mla.c` — `dsv2_mla_forward_train` saves activations; `dsv2_mla_backward`
3. `model.c` — `dsv2_model_train_step_1layer` chains RMSNorm + MLA

**Full model C training (all MoE weights):** use notebook 15 PyTorch `Trainer` for now, export with `scripts/export_v2_tiny.py`, sample in C.
